In [26]:
import mlflow.pyfunc
from pathlib import Path
import subprocess
import os

class ToolsCheck(mlflow.pyfunc.PythonModel):
    def predict(self, context, model_input):
        base_path = Path(context.artifacts["colmap_bundle"])
        results = []

        def check_tool(name, binary_name, args=None):
            bin_path = base_path / name / "bin" / binary_name
            lib_path = base_path / name / "lib"

            env = os.environ.copy()
            env["LD_LIBRARY_PATH"] = f"{lib_path}:{env.get('LD_LIBRARY_PATH', '')}"

            cmd = [str(bin_path)]
            if args:
                cmd.extend(args)

            try:
                result = subprocess.run(
                    cmd,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                    env=env,
                    check=True
                )
                return {
                    "tool": binary_name,
                    "ok": True,
                    "version": result.stdout.decode().strip()
                }
            except subprocess.CalledProcessError as e:
                return {
                    "tool": binary_name,
                    "ok": False,
                    "stderr": e.stderr.decode()
                }
            except FileNotFoundError as e:
                return {
                    "tool": binary_name,
                    "ok": False,
                    "stderr": str(e)
                }

        # ✅ Check standard tools
        results.append(check_tool("colmap", "colmap", ["-h"]))
        results.append(check_tool("ffmpeg", "ffmpeg", ["-version"]))
        results.append(check_tool("imagemagick", "convert", ["-version"]))
        results.append(check_tool("brush", "brush_app", ["--help"]))

        # ✅ Vulkan check
        vulkan_bin = base_path / "vulkan" / "bin" / "vulkaninfo"
        vulkan_lib = base_path / "vulkan" / "lib"
        icd_file = base_path / "vulkan" / "icd.d" / "nvidia_icd.json"

        if vulkan_bin.exists() and icd_file.exists():
            env = os.environ.copy()
            env["LD_LIBRARY_PATH"] = f"{vulkan_lib}:{env.get('LD_LIBRARY_PATH', '')}"
            env["VK_ICD_FILENAMES"] = str(icd_file)

            try:
                result = subprocess.run(
                    [str(vulkan_bin)],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                    env=env,
                    check=True
                )

                grep = subprocess.run(
                    ["grep", "deviceName"],
                    input=result.stdout,
                    text=True,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE
                )

                results.append({
                    "tool": "vulkaninfo",
                    "ok": True,
                    "device": grep.stdout.strip(),
                    "raw": result.stdout.decode(errors="ignore")
                })

            except subprocess.CalledProcessError as e:
                results.append({
                    "tool": "vulkaninfo",
                    "ok": False,
                    "stderr": e.stderr.decode(errors="ignore"),
                    "raw": e.stdout.decode(errors="ignore") if hasattr(e, "stdout") else ""
                })

        else:
            results.append({
                "tool": "vulkaninfo",
                "ok": False,
                "stderr": "vulkaninfo or ICD file missing"
            })

        return results


In [27]:
import mlflow
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec

mlflow.set_tracking_uri("/phoenix/mlflow")
mlflow.set_experiment("Mira3D_ToolCheck")

input_schema = Schema([ColSpec("string", "dummy_input")])
signature = ModelSignature(inputs=input_schema)

with mlflow.start_run(run_name="colmap_check") as run:
    mlflow.pyfunc.log_model(
        artifact_path="mira3d_toolcheck",
        python_model=ToolsCheck(), 
        signature=signature,
        artifacts={"colmap_bundle": "../export"}
    )

    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/mira3d_toolcheck",
        name="mira3d_toolcheck"
    )


Registered model 'mira3d_toolcheck' already exists. Creating a new version of this model...
Created version '28' of model 'mira3d_toolcheck'.
